# Classification with NAMpy

This notebook demonstrates how to use NAMpy for binary and multi-class classification tasks.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_breast_cancer, load_iris, make_classification
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    roc_curve,
)
from sklearn.model_selection import train_test_split

# Import NAMpy models
from nampy.models import NAMClassifier, NBMClassifier

np.random.seed(42)

## 1. Binary Classification

Let's start with binary classification using the Breast Cancer Wisconsin dataset.

In [ ]:
# Load Breast Cancer dataset
cancer = load_breast_cancer()
X_cancer = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_cancer = cancer.target

print(f"Dataset shape: {X_cancer.shape}")
print(f"Classes: {cancer.target_names}")
print("\nClass distribution:")
print(pd.Series(y_cancer).value_counts())

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Train NAM Classifier
model = NAMClassifier(
    numerical_preprocessing="standardization",
    dropout=0.1,
    layer_sizes=[64, 32, 16],
)

model.fit(
    X_train,
    y_train,
    max_epochs=100,
    lr=1e-3,
    patience=10,
    batch_size=64
)

In [ ]:
# Make predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

In [ ]:
# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=cancer.target_names, yticklabels=cancer.target_names)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba[:, 1])
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 2. Feature Importance via Shape Functions

NAMs provide interpretability through shape functions. Let's visualize the most important features.

In [ ]:
# Get shape function outputs
shape_outputs = model.get_shape_function_outputs(X_test)

# Calculate feature importance as variance of shape function outputs
feature_importance = np.var(shape_outputs, axis=0)
importance_df = pd.DataFrame({
    'feature': cancer.feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

# Plot top 10 features
plt.figure(figsize=(10, 6))
top_10 = importance_df.head(10)
plt.barh(range(len(top_10)), top_10['importance'].values, color='steelblue', edgecolor='black')
plt.yticks(range(len(top_10)), top_10['feature'].values)
plt.xlabel('Feature Importance (Shape Function Variance)')
plt.title('Top 10 Most Important Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Plot shape functions for top 4 features
top_features = importance_df.head(4)['feature'].values
top_indices = [list(cancer.feature_names).index(f) for f in top_features]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, feat, idx in zip(axes, top_features, top_indices):
    sort_idx = np.argsort(X_test[feat].values)
    x_sorted = X_test[feat].values[sort_idx]
    y_sorted = shape_outputs[:, idx][sort_idx]

    # Color by actual class
    colors = ['red' if y == 0 else 'blue' for y in y_test[sort_idx]]
    ax.scatter(x_sorted, y_sorted, c=colors, alpha=0.5, s=20)
    ax.set_xlabel(feat)
    ax.set_ylabel('Shape Function Output')
    ax.set_title(f'Shape Function: {feat}')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Shape Functions for Top Features\n(Red=Malignant, Blue=Benign)', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Multi-class Classification

NAMpy also supports multi-class classification. Let's try it on the Iris dataset.

In [ ]:
# Load Iris dataset
iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = iris.target

print(f"Dataset shape: {X_iris.shape}")
print(f"Classes: {iris.target_names}")
print("\nClass distribution:")
print(pd.Series(y_iris).value_counts().sort_index())

In [ ]:
# Split data
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

# Train model
model_iris = NAMClassifier(
    numerical_preprocessing="ple",
    n_bins=20,
    dropout=0.1,
    layer_sizes=[32, 16],
)

model_iris.fit(
    X_train_iris,
    y_train_iris,
    max_epochs=100,
    lr=1e-3,
    patience=10,
    batch_size=32
)

In [ ]:
# Evaluate
y_pred_iris = model_iris.predict(X_test_iris)
accuracy_iris = accuracy_score(y_test_iris, y_pred_iris)

print(f"Accuracy: {accuracy_iris:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_iris, y_pred_iris, target_names=iris.target_names))

In [ ]:
# Confusion matrix
plt.figure(figsize=(8, 6))
cm_iris = confusion_matrix(y_test_iris, y_pred_iris)
sns.heatmap(cm_iris, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Multi-class Classification: Iris Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize shape functions for Iris
shape_outputs_iris = model_iris.get_shape_function_outputs(X_test_iris)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

colors = ['red', 'green', 'blue']
for ax, col, i in zip(axes, iris.feature_names, range(4)):
    for class_idx in range(3):
        mask = y_test_iris == class_idx
        ax.scatter(
            X_test_iris[col].values[mask],
            shape_outputs_iris[:, i][mask],
            c=colors[class_idx],
            alpha=0.6,
            label=iris.target_names[class_idx],
            s=30
        )
    ax.set_xlabel(col)
    ax.set_ylabel('Shape Function Output')
    ax.set_title(f'Shape Function: {col}')
    ax.legend()
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Shape Functions for Iris Dataset', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Synthetic Multi-class Example

Let's create a more complex multi-class classification problem.

In [ ]:
# Generate synthetic multi-class data
X_synth, y_synth = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_classes=4,
    n_clusters_per_class=1,
    random_state=42
)

X_synth = pd.DataFrame(X_synth, columns=[f'feature_{i}' for i in range(10)])

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_synth, y_synth, test_size=0.2, random_state=42, stratify=y_synth
)

print(f"Dataset shape: {X_synth.shape}")
print(f"Number of classes: {len(np.unique(y_synth))}")

In [ ]:
# Compare NAMClassifier and NBMClassifier
models_synth = {
    'NAM': NAMClassifier(numerical_preprocessing='ple', n_bins=30, dropout=0.1),
    'NBM': NBMClassifier(numerical_preprocessing='ple', n_bins=30, dropout=0.1),
}

results_synth = {}
for name, mdl in models_synth.items():
    print(f"\nTraining {name}...")
    mdl.fit(X_train_s, y_train_s, max_epochs=50, lr=1e-3, patience=5, batch_size=128)

    y_pred = mdl.predict(X_test_s)
    acc = accuracy_score(y_test_s, y_pred)

    results_synth[name] = acc
    print(f"  {name} Accuracy: {acc:.4f}")

In [ ]:
# Visualize results
plt.figure(figsize=(8, 5))
plt.bar(results_synth.keys(), results_synth.values(), color=['steelblue', 'coral'], edgecolor='black')
plt.ylabel('Accuracy')
plt.title('Model Comparison: 4-class Classification')
plt.ylim(0, 1)
for i, (_name, acc) in enumerate(results_synth.items()):
    plt.text(i, acc + 0.02, f'{acc:.4f}', ha='center')
plt.tight_layout()
plt.show()

## Summary

In this notebook, we learned how to:

1. **Perform binary classification** with NAMClassifier
2. **Evaluate models** using accuracy, classification report, confusion matrix, and ROC curves
3. **Interpret feature importance** through shape function analysis
4. **Handle multi-class classification** problems
5. **Compare different NAMpy classifier models**

NAMpy classifiers provide interpretable predictions while maintaining competitive accuracy with black-box models.